# Jet Tagging Solution

Course project for **Physics Applications of AI**. This notebook solves the jet-tagging assignment for ATLAS-like jets using reusable code from `src/physics_applications_of_ai/`.

The project studies three tasks: binary classification, three-class classification, and momentum-decorrelated classification. The original assignment handout is preserved separately in `instructions.ipynb`.

## Approach

The solution uses gradient-boosted decision trees on engineered jet features. Tasks 1 and 2 use global substructure variables plus relative constituent features where useful. Task 3 removes direct four-momentum inputs, balances examples in shared `(pt, mass)` bins, and audits residual momentum dependence.

Run settings live in `physics_applications_of_ai.config`. Set `SMOKE_RUN = True` in the setup cell for a fast smoke run; keep it `False` for the full project run. Metrics and diagnostic plots are written to `outputs/`.

In [ ]:
import pandas
from matplotlib import pyplot
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report
from sklearn.model_selection import train_test_split

from physics_applications_of_ai.artifacts import save_figure, save_table
from physics_applications_of_ai.config import FULL_RUN_SETTINGS, QUICK_RUN_SETTINGS
from physics_applications_of_ai.data import DISPLAY_LABELS
from physics_applications_of_ai.datasets import (
    make_binary_dataset,
    make_decorrelated_dataset,
    make_multiclass_dataset,
)
from physics_applications_of_ai.evaluation import (
    binary_metrics,
    multiclass_metrics,
    prediction_momentum_eta_squared,
    probability_momentum_correlations,
)
from physics_applications_of_ai.models import make_hist_gradient_boosting_classifier

SMOKE_RUN = False
RUN_SETTINGS = QUICK_RUN_SETTINGS if SMOKE_RUN else FULL_RUN_SETTINGS


def train_classifier(X, y, *, classifier_settings):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=RUN_SETTINGS.test_size,
        stratify=y,
        random_state=RUN_SETTINGS.random_state,
    )
    model = make_hist_gradient_boosting_classifier(
        max_iter=classifier_settings.max_iter,
        learning_rate=classifier_settings.learning_rate,
        l2_regularization=classifier_settings.l2_regularization,
        random_state=RUN_SETTINGS.random_state,
    )
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)
    predictions = probabilities.argmax(axis=1)
    return model, X_test, y_test, predictions, probabilities

## Task 1: Binary Classification

Two balanced binary classifiers are trained: quark/gluon vs W/Z and quark/gluon vs top. The table reports accuracy, F1, ROC-AUC, and average precision on held-out test splits.

This is the easiest setting because each model only needs to distinguish one signal class from the quark/gluon background. ROC-AUC and average precision are especially useful here because they describe the quality of the classifier ranking across thresholds, rather than only the default 0.5 decision threshold.

In [ ]:
task1_evaluations = []

for positive_label, display_label in [("wz", "W/Z"), ("top", "top")]:
    X, y = make_binary_dataset(
        positive_label,
        n_per_class=RUN_SETTINGS.binary_n_per_class,
        random_state=RUN_SETTINGS.random_state,
    )
    model, X_test, y_test, _, probabilities = train_classifier(
        X, y, classifier_settings=RUN_SETTINGS.binary_classifier
    )
    positive_probabilities = probabilities[:, 1]
    predictions = (positive_probabilities >= 0.5).astype(int)

    print(f"\nquark/gluon vs {display_label}")
    print(classification_report(y_test, predictions, target_names=["quark/gluon", display_label]))

    task1_evaluations.append({
        "task": f"quark/gluon vs {display_label}",
        "model": model,
        "X_test": X_test,
        "y_test": y_test,
        "predictions": predictions,
        "probabilities": positive_probabilities,
        **binary_metrics(y_test, predictions, positive_probabilities),
    })

task1_results = pandas.DataFrame([
    {metric: evaluation[metric] for metric in ["task", "accuracy", "f1", "roc_auc", "average_precision"]}
    for evaluation in task1_evaluations
]).set_index("task")
save_table(task1_results, RUN_SETTINGS.output_dir, "task1_metrics.csv")
task1_results

In [ ]:
fig, axes = pyplot.subplots(2, 2, figsize=(12, 9))
for row_index, evaluation in enumerate(task1_evaluations):
    positive_label = evaluation["task"].split(" vs ")[1]
    ConfusionMatrixDisplay.from_predictions(
        evaluation["y_test"],
        evaluation["predictions"],
        display_labels=["quark/gluon", positive_label],
        normalize="true",
        ax=axes[row_index, 0],
        colorbar=False,
    )
    axes[row_index, 0].set_title(f"{evaluation['task']} confusion matrix")

    RocCurveDisplay.from_predictions(
        evaluation["y_test"], evaluation["probabilities"], ax=axes[row_index, 1]
    )
    axes[row_index, 1].set_title(f"{evaluation['task']} ROC curve")

pyplot.tight_layout()
save_figure(fig, RUN_SETTINGS.output_dir, "task1_binary_diagnostics.png")

## Task 2: Multi-Class Classification

The multi-class model uses a balanced sample from all three classes and combines global substructure features with relative constituent features.

This task is harder than Task 1 because the model must separate all three jet origins simultaneously. The normalized confusion matrix should be read as part of the result: it shows which classes remain physically or feature-wise similar after training, not just the overall accuracy.

In [ ]:
X_task2, y_task2 = make_multiclass_dataset(
    n_per_class=RUN_SETTINGS.multiclass_n_per_class,
    random_state=RUN_SETTINGS.random_state,
)
task2_model, X_task2_test, y_task2_test, task2_predictions, task2_probabilities = train_classifier(
    X_task2, y_task2, classifier_settings=RUN_SETTINGS.multiclass_classifier
)

print(classification_report(y_task2_test, task2_predictions, target_names=DISPLAY_LABELS))

task2_results = pandas.DataFrame([
    multiclass_metrics(y_task2_test, task2_predictions, task2_probabilities)
])
save_table(task2_results, RUN_SETTINGS.output_dir, "task2_metrics.csv")
task2_results

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_task2_test,
    task2_predictions,
    display_labels=DISPLAY_LABELS,
    normalize="true",
    cmap="Blues",
)
pyplot.title("Task 2 normalized confusion matrix")
pyplot.tight_layout()
save_figure(pyplot.gcf(), RUN_SETTINGS.output_dir, "task2_confusion_matrix.png")

## Task 3: Momentum Decorrelation

The decorrelated model omits direct `pt`, `eta`, `phi`, and `mass` inputs. It uses unitless substructure ratios, relative constituent geometry, and equalized sampling in shared `(pt, mass)` bins. The metric tables show both classification performance and residual momentum dependence.

The intended outcome is not simply maximum accuracy. A useful decorrelated tagger should reduce dependence on global momentum variables even if its classification metrics are lower than the standard multiclass model. The probability-correlation table and eta-squared audit quantify this tradeoff.

In [ ]:
X_task3, y_task3, task3_momenta = make_decorrelated_dataset(
    n_bins=RUN_SETTINGS.decorrelated_n_bins,
    max_per_bin_per_class=RUN_SETTINGS.decorrelated_max_per_bin_per_class,
    min_per_bin_per_class=RUN_SETTINGS.decorrelated_min_per_bin_per_class,
    random_state=RUN_SETTINGS.random_state,
)
(
    X_task3_train,
    X_task3_test,
    y_task3_train,
    y_task3_test,
    task3_momenta_train,
    task3_momenta_test,
) = train_test_split(
    X_task3,
    y_task3,
    task3_momenta,
    test_size=RUN_SETTINGS.test_size,
    stratify=y_task3,
    random_state=RUN_SETTINGS.random_state,
)

task3_model = make_hist_gradient_boosting_classifier(
    max_iter=RUN_SETTINGS.decorrelated_classifier.max_iter,
    learning_rate=RUN_SETTINGS.decorrelated_classifier.learning_rate,
    l2_regularization=RUN_SETTINGS.decorrelated_classifier.l2_regularization,
    random_state=RUN_SETTINGS.random_state,
)
task3_model.fit(X_task3_train, y_task3_train)
task3_probabilities = task3_model.predict_proba(X_task3_test)
task3_predictions = task3_probabilities.argmax(axis=1)

print(classification_report(y_task3_test, task3_predictions, target_names=DISPLAY_LABELS))

task3_results = pandas.DataFrame([{
    **multiclass_metrics(y_task3_test, task3_predictions, task3_probabilities),
    "n_train": len(y_task3_train),
    "n_test": len(y_task3_test),
    "n_features": X_task3.shape[1],
}])
task3_probability_momentum_correlations = probability_momentum_correlations(
    task3_probabilities, task3_momenta_test
)
task3_prediction_momentum_eta_squared = prediction_momentum_eta_squared(
    task3_predictions, task3_momenta_test
)
save_table(task3_results, RUN_SETTINGS.output_dir, "task3_metrics.csv")
save_table(
    task3_probability_momentum_correlations,
    RUN_SETTINGS.output_dir,
    "task3_probability_momentum_correlations.csv",
)
save_table(
    task3_prediction_momentum_eta_squared,
    RUN_SETTINGS.output_dir,
    "task3_prediction_momentum_eta_squared.csv",
)

display(task3_results)
display(task3_probability_momentum_correlations)
display(task3_prediction_momentum_eta_squared)

In [ ]:
fig, axes = pyplot.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    y_task3_test,
    task3_predictions,
    display_labels=DISPLAY_LABELS,
    normalize="true",
    cmap="Blues",
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("Task 3 normalized confusion matrix")

task3_momenta_test.assign(predicted_class=task3_predictions).boxplot(
    column="mass", by="predicted_class", ax=axes[1]
)
axes[1].set_title("Jet mass by predicted class after decorrelation")
axes[1].set_xlabel("predicted class index")
axes[1].set_ylabel("mass [GeV]")
fig.suptitle("")
pyplot.tight_layout()
save_figure(fig, RUN_SETTINGS.output_dir, "task3_decorrelation_diagnostics.png")

## Summary

The binary classifiers are expected to perform strongest because they solve simpler one-vs-one-style problems. The three-class classifier is more realistic and correspondingly more difficult, so the confusion matrix is important for interpreting which jet origins overlap.

The decorrelated model demonstrates the main physics tradeoff in the project. Removing direct momentum inputs and balancing in `(pt, mass)` bins should reduce kinematic dependence, but it also removes information that helps classification. The final model should therefore be judged by both tagging performance and the residual momentum-dependence audits, not by accuracy alone.